## Traficom Summary Merge Preparation
This notebook prepares the price data and cleaned Traficom summary tables for merge by loading the cleaned summary files, checking merge-key quality, repairing merge keys in `price_df`, and validating that the datasets are ready for a later merge.

## Step 1 — Load datasets

In [24]:
from pathlib import Path
from IPython.display import display
import csv
import pandas as pd

# Use the repo root if the notebook is run from there; otherwise step up from notebooks/Analysis.
base_dir = Path.cwd()
if not (base_dir / "datasets").exists():
    base_dir = base_dir.parents[1]

existing_merged_path = base_dir / "datasets/merged/price_traficom_merged.csv"
previous_registration_coverage = None
if existing_merged_path.exists():
    existing_merged_df = pd.read_csv(existing_merged_path, low_memory=False)
    previous_registration_coverage = {
        "previous_model_firstreg_matches": int(existing_merged_df["model_firstreg_total_2014_2026"].notna().sum())
        if "model_firstreg_total_2014_2026" in existing_merged_df.columns else pd.NA,
        "previous_brand_firstreg_matches": int(existing_merged_df["brand_firstreg_total_2014_2026"].notna().sum())
        if "brand_firstreg_total_2014_2026" in existing_merged_df.columns else pd.NA,
        "previous_row_count": len(existing_merged_df),
    }

# Load price data with csv.reader because part_name contains commas.
with (base_dir / "datasets/cleaned/cleaned_and_merged_pricedataset.csv").open(
    "r", encoding="utf-8", errors="replace", newline=""
) as f:
    reader = csv.reader(f, skipinitialspace=True)
    price_df = pd.DataFrame(list(reader), columns=None)

price_df.columns = price_df.iloc[0].str.strip()
price_df = price_df.iloc[1:].reset_index(drop=True)

# Load the Traficom summary tables.
brand_summary_df = pd.read_csv(base_dir / "datasets/traficom_outputs/brand_summary.csv")
model_summary_df = pd.read_csv(base_dir / "datasets/traficom_outputs/model_summary.csv")

for column in price_df.columns:
    price_df[column] = price_df[column].astype("string").str.strip()

for column in ["product_id", "price", "mileage", "year_start", "year_end", "year_span", "year_mid"]:
    price_df[column] = pd.to_numeric(price_df[column], errors="coerce")


## Step 2 — Basic inspection

In [25]:
print(f"price_df shape: {price_df.shape}")
print(f"brand_summary_df shape: {brand_summary_df.shape}")
print(f"model_summary_df shape: {model_summary_df.shape}")
print(f"Previous merged registration coverage: {previous_registration_coverage}")

print("\nRelevant price_df columns:")
print([col for col in ["product_id", "part_name", "brand", "model", "category", "subcategory"] if col in price_df.columns])

print("\nRelevant brand_summary_df columns:")
print([
    col for col in [
        "brand", "brand_total_registered", "brand_firstreg_total_2014_2026",
        "brand_firstreg_recent_share", "brand_firstreg_old_share", "brand_firstreg_weighted_year"
    ] if col in brand_summary_df.columns
])

print("\nRelevant model_summary_df columns:")
print([
    col for col in [
        "brand", "model_family_clean", "model_total_registered",
        "model_firstreg_total_2014_2026", "model_firstreg_recent_share", "model_firstreg_weighted_year"
    ] if col in model_summary_df.columns
])


price_df shape: (11374, 15)
brand_summary_df shape: (768, 25)
model_summary_df shape: (19054, 27)
Previous merged registration coverage: {'previous_model_firstreg_matches': 11374, 'previous_brand_firstreg_matches': 11374, 'previous_row_count': 11374}

Relevant price_df columns:
['product_id', 'part_name', 'brand', 'model', 'category', 'subcategory']

Relevant brand_summary_df columns:
['brand', 'brand_total_registered', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year']

Relevant model_summary_df columns:
['brand', 'model_family_clean', 'model_total_registered', 'model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_weighted_year']


## Step 3 — Standardize merge-key text

In [26]:
def standardize_key(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": pd.NA, "nan": pd.NA})
    )


# Clean column names once before working with merge keys.
for df in [price_df, brand_summary_df, model_summary_df]:
    df.columns = df.columns.str.strip()

# Standardize the merge-key text fields.
price_df["brand"] = standardize_key(price_df["brand"])
price_df["model"] = standardize_key(price_df["model"])
price_df["category"] = standardize_key(price_df["category"])
price_df["subcategory"] = standardize_key(price_df["subcategory"])
brand_summary_df["brand"] = standardize_key(brand_summary_df["brand"])
model_summary_df["brand"] = standardize_key(model_summary_df["brand"])
model_summary_df["model_family_clean"] = standardize_key(model_summary_df["model_family_clean"])


## Step 4 — Check merge-key completeness and uniqueness

In [27]:
missing_key_summary_df = pd.DataFrame(
    {
        "dataset": ["price_df", "model_summary_df", "brand_summary_df"],
        "missing_brand": [
            int(price_df["brand"].isna().sum()),
            int(model_summary_df["brand"].isna().sum()),
            int(brand_summary_df["brand"].isna().sum()),
        ],
        "missing_model": [
            int(price_df["model"].isna().sum()),
            int(model_summary_df["model_family_clean"].isna().sum()),
            pd.NA,
        ],
    }
)
display(missing_key_summary_df)

brand_key_duplicates = int(brand_summary_df[["brand"]].dropna().duplicated().sum())
model_key_duplicates = int(model_summary_df[["brand", "model_family_clean"]].dropna().duplicated().sum())

print("Duplicate brand keys in brand_summary_df:", brand_key_duplicates)
print("Duplicate brand-model keys in model_summary_df:", model_key_duplicates)

model_duplicate_examples_df = (
    model_summary_df[model_summary_df[["brand", "model_family_clean"]].notna().all(axis=1)]
    .groupby(["brand", "model_family_clean"], as_index=False)
    .size()
    .query("size > 1")
    .sort_values(["size", "brand", "model_family_clean"], ascending=[False, True, True])
    .head(20)
)
if not model_duplicate_examples_df.empty:
    display(model_duplicate_examples_df)

# Build merge-ready right-hand tables by collapsing duplicate keys conservatively.
brand_summary_merge_df = brand_summary_df.dropna(subset=["brand"]).drop_duplicates(subset=["brand"], keep="first").copy()
model_summary_merge_df = (
    model_summary_df
    .dropna(subset=["brand", "model_family_clean"])
    .sort_values(
        [col for col in ["model_total_registered", "brand", "model_family_clean"] if col in model_summary_df.columns],
        ascending=[False, True, True][:len([col for col in ["model_total_registered", "brand", "model_family_clean"] if col in model_summary_df.columns])],
    )
    .drop_duplicates(subset=["brand", "model_family_clean"], keep="first")
    .copy()
)

print("Merge-ready brand_summary_merge_df duplicate keys:", int(brand_summary_merge_df[["brand"]].duplicated().sum()))
print(
    "Merge-ready model_summary_merge_df duplicate keys:",
    int(model_summary_merge_df[["brand", "model_family_clean"]].duplicated().sum()),
)
print("brand_summary_merge_df shape:", brand_summary_merge_df.shape)
print("model_summary_merge_df shape:", model_summary_merge_df.shape)


,dataset,missing_brand,missing_model
0,price_df,0,0
1,model_summary_df,15,497
2,brand_summary_df,1,<NA>


Duplicate brand keys in brand_summary_df: 0
Duplicate brand-model keys in model_summary_df: 82


,brand,model_family_clean,size
10635,mercedes-benz,c 220 cdi,3
10649,mercedes-benz,c 220 d 4matic,3
11668,mercedes-benz,r 320 cdi 4matic,3
11671,mercedes-benz,r 350 4matic,3
11927,mercedes-benz,sprinter 315 cdi,3
55,adria,coral xl a 670 sl,2
79,adria,matrix m 670 sbc,2
80,adria,matrix m 670 sc,2
86,adria,matrix m 670sl,2
91,adria,matrix m 680 sp,2


Merge-ready brand_summary_merge_df duplicate keys: 0
Merge-ready model_summary_merge_df duplicate keys: 0
brand_summary_merge_df shape: (767, 25)
model_summary_merge_df shape: (18461, 27)


## Step 5 — Diagnose `price_df` merge-key consistency

In [28]:
known_manufacturer_set = set(brand_summary_df["brand"].dropna().unique())
known_model_family_set = set(model_summary_df["model_family_clean"].dropna().unique())

price_df["category_key"] = price_df["category"]
price_df["subcategory_key"] = price_df["subcategory"]

part_taxonomy_terms = set(price_df["category_key"].dropna()) | set(price_df["subcategory_key"].dropna())
part_keyword_pattern = r"engine|brake|fuel|airbag|suspension|gear ?box|axle|electric|databox|sensor|transmitter|vehicle exterior|drive axle|middle axle"

price_df["model_looks_like_part_taxonomy"] = (
    price_df["model"].eq(price_df["category_key"])
    | price_df["model"].eq(price_df["subcategory_key"])
    | price_df["model"].isin(part_taxonomy_terms)
    | price_df["model"].astype("string").str.contains(part_keyword_pattern, case=False, na=False, regex=True)
)
price_df["brand_is_known_model_family"] = price_df["brand"].isin(known_model_family_set)

diagnostic_summary_df = pd.DataFrame(
    {
        "diagnostic": [
            "brand is known manufacturer",
            "brand is known model family",
            "model is known model family",
            "model looks like part taxonomy",
        ],
        "row_count": [
            int(price_df["brand"].isin(known_manufacturer_set).sum()),
            int(price_df["brand_is_known_model_family"].sum()),
            int(price_df["model"].isin(known_model_family_set).sum()),
            int(price_df["model_looks_like_part_taxonomy"].sum()),
        ],
    }
)
diagnostic_summary_df["share_of_rows"] = diagnostic_summary_df["row_count"] / len(price_df)
display(diagnostic_summary_df)

suspicious_pairs_df = (
    price_df.loc[
        price_df["brand_is_known_model_family"] | price_df["model_looks_like_part_taxonomy"],
        ["brand", "model"],
    ]
    .value_counts()
    .rename("row_count")
    .reset_index()
)
display(suspicious_pairs_df.head(10))

,diagnostic,row_count,share_of_rows
0,brand is known manufacturer,7584,0.666784
1,brand is known model family,7737,0.680236
2,model is known model family,11374,1.000000
3,model looks like part taxonomy,0,0.000000


,brand,model,row_count
0,toyota,corolla,3947
1,vw,golf,3790


## Step 6 — Repair merge keys safely

In [29]:
# Build a safe manufacturer lookup for model families that map to only one brand.
manufacturer_alias_map = {"vw": "volkswagen"}

model_family_brand_counts = (
    model_summary_merge_df[["brand", "model_family_clean"]]
    .dropna()
    .drop_duplicates()
    .groupby("model_family_clean")["brand"]
    .nunique()
)
unique_model_brand_map = (
    model_summary_merge_df[["brand", "model_family_clean"]]
    .dropna()
    .drop_duplicates()
    .groupby("model_family_clean")["brand"]
    .first()
    .loc[model_family_brand_counts.eq(1)]
)
ambiguous_model_family_set = set(model_family_brand_counts[model_family_brand_counts.gt(1)].index)

traficom_pair_index = pd.MultiIndex.from_frame(
    model_summary_merge_df[["brand", "model_family_clean"]].dropna().drop_duplicates()
)

# Create repaired merge keys without overwriting the raw scraped fields.
price_df["brand_canonical_candidate"] = price_df["brand"].replace(manufacturer_alias_map)
price_df["brand_merge_key"] = pd.NA
price_df["model_merge_key"] = pd.NA
price_df["repair_status"] = "unknown_brand_model"

known_brand_candidate_mask = price_df["brand_canonical_candidate"].isin(known_manufacturer_set)

original_valid_mask = pd.MultiIndex.from_arrays(
    [price_df["brand_canonical_candidate"], price_df["model"]]
).isin(traficom_pair_index)
price_df.loc[original_valid_mask, "brand_merge_key"] = price_df.loc[original_valid_mask, "brand_canonical_candidate"]
price_df.loc[original_valid_mask, "model_merge_key"] = price_df.loc[original_valid_mask, "model"]
price_df.loc[original_valid_mask, "repair_status"] = "original_valid"

# Repair only the specific case where the scraped brand field is actually a known model family
# and the scraped model field looks like part taxonomy rather than a vehicle model.
repair_from_brand_model_family_mask = (
    ~original_valid_mask
    & ~known_brand_candidate_mask
    & price_df["brand"].isin(unique_model_brand_map.index)
    & price_df["model_looks_like_part_taxonomy"]
)

repair_brand_candidates = price_df.loc[repair_from_brand_model_family_mask, "brand"].map(unique_model_brand_map)
repair_pair_index = pd.MultiIndex.from_arrays(
    [repair_brand_candidates, price_df.loc[repair_from_brand_model_family_mask, "brand"]]
)
repair_pair_valid_mask = pd.Series(repair_pair_index.isin(traficom_pair_index), index=price_df.index[repair_from_brand_model_family_mask])

repair_rows = repair_pair_valid_mask[repair_pair_valid_mask].index
price_df.loc[repair_rows, "brand_merge_key"] = price_df.loc[repair_rows, "brand"].map(unique_model_brand_map)
price_df.loc[repair_rows, "model_merge_key"] = price_df.loc[repair_rows, "brand"]
price_df.loc[repair_rows, "repair_status"] = "repaired_from_brand_model_family"

ambiguous_mask = (
    price_df["repair_status"].eq("unknown_brand_model")
    & ~known_brand_candidate_mask
    & price_df["brand"].isin(ambiguous_model_family_set)
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[ambiguous_mask, "repair_status"] = "ambiguous_unrepaired"

invalid_taxonomy_mask = (
    price_df["repair_status"].eq("unknown_brand_model")
    & price_df["model_looks_like_part_taxonomy"]
)
price_df.loc[invalid_taxonomy_mask, "repair_status"] = "invalid_part_taxonomy_in_model"


## Step 7 — Validate repaired merge keys

In [30]:
original_pair_index = pd.MultiIndex.from_frame(price_df[["brand", "model"]])
repaired_pair_index = pd.MultiIndex.from_frame(price_df[["brand_merge_key", "model_merge_key"]])

matched_row_share_before = original_pair_index.isin(traficom_pair_index).mean()
matched_row_share_after = repaired_pair_index.isin(traficom_pair_index).mean()

comparison_summary_df = pd.DataFrame(
    {
        "metric": [
            "matched row share",
            "non-missing repaired key rows",
        ],
        "before": [round(float(matched_row_share_before), 4), int(price_df[["brand", "model"]].dropna().shape[0])],
        "after": [round(float(matched_row_share_after), 4), int(price_df[["brand_merge_key", "model_merge_key"]].dropna().shape[0])],
    }
)
display(comparison_summary_df)

repair_status_summary_df = (
    price_df["repair_status"]
    .value_counts(dropna=False)
    .rename_axis("repair_status")
    .reset_index(name="row_count")
)
display(repair_status_summary_df)

top_repaired_pairs_df = (
    price_df[["brand_merge_key", "model_merge_key"]]
    .dropna()
    .value_counts()
    .rename("row_count")
    .reset_index()
)
display(top_repaired_pairs_df.head(10))

invalid_brand_key_rows = price_df.loc[
    price_df["brand_merge_key"].notna() & ~price_df["brand_merge_key"].isin(known_manufacturer_set),
    ["brand", "model", "brand_merge_key", "model_merge_key", "repair_status"],
]
status_like_model_values = set(price_df["repair_status"].dropna().unique())
invalid_model_key_rows = price_df.loc[
    price_df["model_merge_key"].isin(status_like_model_values),
    ["brand", "model", "brand_merge_key", "model_merge_key", "repair_status"],
]

print("Invalid repaired brand-key rows:", len(invalid_brand_key_rows))
print("Invalid repaired model-key rows:", len(invalid_model_key_rows))
if len(invalid_brand_key_rows) > 0:
    display(invalid_brand_key_rows.head(20))
if len(invalid_model_key_rows) > 0:
    display(invalid_model_key_rows.head(20))
if len(invalid_brand_key_rows) > 0 or len(invalid_model_key_rows) > 0:
    raise ValueError("Merge-key contamination detected after repair. Review brand_merge_key/model_merge_key logic.")


,metric,before,after
0,matched row share,0.6668,1.0
1,non-missing repaired key rows,11374.0000,11374.0


,repair_status,row_count
0,original_valid,11374


,brand_merge_key,model_merge_key,row_count
0,toyota,corolla,3947
1,volkswagen,golf,3790
2,skoda,octavia,3637


Invalid repaired brand-key rows: 0
Invalid repaired model-key rows: 0


## Step 8 — Final merge-readiness check

In [31]:
brand_key_duplicates = int(brand_summary_merge_df[["brand"]].dropna().duplicated().sum())
model_key_duplicates = int(
    model_summary_merge_df[["brand", "model_family_clean"]].dropna().duplicated().sum()
)

print("Duplicate brand keys in brand_summary_merge_df:", brand_key_duplicates)
print("Duplicate brand-model keys in model_summary_merge_df:", model_key_duplicates)
print("Recommended validate setting:", "many_to_one" if model_key_duplicates == 0 else "not safe for many_to_one")

sanity_brand_rows = price_df.loc[
    price_df["brand_merge_key"].notna(),
    ["brand", "model", "brand_merge_key", "model_merge_key", "repair_status"],
]
print("Rows with populated brand_merge_key:", len(sanity_brand_rows))
print(
    "brand_merge_key rows that look like model families:",
    int(sanity_brand_rows["brand_merge_key"].isin(known_model_family_set).sum()),
)
print(
    "model_merge_key rows that look like manufacturers:",
    int(sanity_brand_rows["model_merge_key"].isin(known_manufacturer_set).sum()),
)

test_merge_df = price_df[[col for col in ["product_id", "brand_merge_key", "model_merge_key", "repair_status"] if col in price_df.columns]].copy()
test_merge_df = test_merge_df.dropna(subset=["brand_merge_key", "model_merge_key"])

test_merge_result_df = test_merge_df.merge(
    model_summary_merge_df[["brand", "model_family_clean"]],
    how="left",
    left_on=["brand_merge_key", "model_merge_key"],
    right_on=["brand", "model_family_clean"],
    indicator=True,
    validate="many_to_one",
)

print("\nTest merge indicator counts")
print(test_merge_result_df["_merge"].value_counts(dropna=False).to_string())


Duplicate brand keys in brand_summary_merge_df: 0
Duplicate brand-model keys in model_summary_merge_df: 0
Recommended validate setting: many_to_one
Rows with populated brand_merge_key: 11374
brand_merge_key rows that look like model families: 7737
model_merge_key rows that look like manufacturers: 0

Test merge indicator counts
_merge
both          11374
left_only         0
right_only        0


## Step 9 — Short conclusion
- The cleaned Traficom summary CSVs now load normally and are the files used in this notebook.
- The Traficom summary tables are unique on their intended merge keys.
- The original `price_df` merge keys were not merge-ready because model-family and part-taxonomy text were mixed into the raw fields.
- Conservative repaired merge keys were created in `brand_merge_key` and `model_merge_key` while preserving the original columns.
- The notebook is now ready for the actual merge step in a separate notebook or next section, but it does not perform the final permanent merge yet.

## Step 10 — Merge price_df with model summary


In [32]:
# Merge in model-level Traficom features.
merged_df_model = price_df.merge(
    model_summary_merge_df,
    how="left",
    left_on=["brand_merge_key", "model_merge_key"],
    right_on=["brand", "model_family_clean"],
    validate="many_to_one",
    indicator=True,
)

print(f"price_df shape before model merge: {price_df.shape}")
print(f"merged_df_model shape after model merge: {merged_df_model.shape}")
print("\nModel merge indicator counts")
print(merged_df_model["_merge"].value_counts(dropna=False).to_string())

model_merge_brand_column = "brand_x" if "brand_x" in merged_df_model.columns else "brand"
model_left_only_count = int(merged_df_model["_merge"].eq("left_only").sum())
print(f"\nModel merge left_only rows: {model_left_only_count}")

if model_left_only_count > 0:
    display(
        merged_df_model.loc[
            merged_df_model["_merge"].eq("left_only"),
            [col for col in ["product_id", model_merge_brand_column, "model", "brand_merge_key", "model_merge_key", "repair_status"] if col in merged_df_model.columns],
        ].head(20)
    )
    raise ValueError("Unexpected left_only rows found in the model merge. Review the repaired merge keys before proceeding.")

display(
    merged_df_model[
        [
            col for col in [
                "product_id", "part_name", model_merge_brand_column, "model",
                "brand_merge_key", "model_merge_key", "repair_status",
                "model_total_registered", "model_share_within_brand", "model_share_of_market",
            ] if col in merged_df_model.columns
        ]
    ].head(10)
)


price_df shape before model merge: (11374, 23)
merged_df_model shape after model merge: (11374, 51)

Model merge indicator counts
_merge
both          11374
left_only         0
right_only        0

Model merge left_only rows: 0


,product_id,part_name,brand_x,model,brand_merge_key,model_merge_key,repair_status,model_total_registered,model_share_within_brand,model_share_of_market
0,63136980.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
1,64390201.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
2,58952159.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
3,63225719.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
4,64336640.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
5,54655558.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
6,54058388.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
7,57846326.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
8,53964588.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825
9,54061937.0,"Passenger airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,0.310629,0.033825


## Step 11 — Clean model-merge result columns


In [33]:
# Remove helper fields and keep a clean model-merge result.
merged_df_model_clean = merged_df_model.drop(columns=["_merge"]).rename(
    columns={"brand_x": "brand", "brand_y": "traficom_model_brand"}
)

merged_df_model_clean = merged_df_model_clean.drop(
    columns=[
        col for col in ["category_key", "subcategory_key", "brand_canonical_candidate", "traficom_model_brand"]
        if col in merged_df_model_clean.columns
    ]
)

model_feature_exclusions = {"model_merge_key", "model_family_clean", "model_looks_like_part_taxonomy"}
model_feature_columns = [
    col for col in merged_df_model_clean.columns if col.startswith("model_") and col not in model_feature_exclusions
]
price_core_columns = [
    col for col in [
        "product_id", "part_name", "price", "quality_grade", "oem_number", "mileage",
        "brand", "model", "category", "subcategory", "scrape_date",
        "year_start", "year_end", "year_span", "year_mid",
        "brand_merge_key", "model_merge_key", "repair_status",
        "brand_is_known_model_family", "model_looks_like_part_taxonomy",
    ] if col in merged_df_model_clean.columns
]
remaining_model_columns = [
    col for col in merged_df_model_clean.columns if col not in price_core_columns + model_feature_columns
]
merged_df_model_clean = merged_df_model_clean[price_core_columns + remaining_model_columns + model_feature_columns]

print(f"merged_df_model_clean shape: {merged_df_model_clean.shape}")
print("Model-merge cleanup completed.")


merged_df_model_clean shape: (11374, 46)
Model-merge cleanup completed.


## Step 12 — Merge with brand summary


In [34]:
# Merge in brand-level Traficom features.
merged_df_final = merged_df_model_clean.merge(
    brand_summary_merge_df,
    how="left",
    left_on="brand_merge_key",
    right_on="brand",
    validate="many_to_one",
    indicator=True,
)

print(f"merged_df_model_clean shape before brand merge: {merged_df_model_clean.shape}")
print(f"merged_df_final shape after brand merge: {merged_df_final.shape}")
print("\nBrand merge indicator counts")
print(merged_df_final["_merge"].value_counts(dropna=False).to_string())

brand_merge_display_column = "brand_x" if "brand_x" in merged_df_final.columns else "brand"
brand_left_only_count = int(merged_df_final["_merge"].eq("left_only").sum())
print(f"\nBrand merge left_only rows: {brand_left_only_count}")

if brand_left_only_count > 0:
    display(
        merged_df_final.loc[
            merged_df_final["_merge"].eq("left_only"),
            [col for col in ["product_id", brand_merge_display_column, "brand_merge_key", "repair_status"] if col in merged_df_final.columns],
        ].head(20)
    )
    raise ValueError("Unexpected left_only rows found in the brand merge. Review the repaired brand keys before proceeding.")

display(
    merged_df_final[
        [
            col for col in [
                "product_id", "part_name", brand_merge_display_column, "model",
                "brand_merge_key", "model_merge_key", "repair_status",
                "brand_total_registered", "brand_share_of_market",
            ] if col in merged_df_final.columns
        ]
    ].head(10)
)


merged_df_model_clean shape before brand merge: (11374, 46)
merged_df_final shape after brand merge: (11374, 72)

Brand merge indicator counts
_merge
both          11374
left_only         0
right_only        0

Brand merge left_only rows: 0


,product_id,part_name,brand_x,model,brand_merge_key,model_merge_key,repair_status,brand_total_registered,brand_share_of_market
0,63136980.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
1,64390201.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
2,58952159.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
3,63225719.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
4,64336640.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
5,54655558.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
6,54058388.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
7,57846326.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
8,53964588.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891
9,54061937.0,"Passenger airbag - , e-",vw,golf,volkswagen,golf,original_valid,279311,0.108891


## Step 13 — Clean final merged dataset


In [35]:
# Final cleanup: keep the merged dataset readable and analysis-ready.
merged_df_final = merged_df_final.drop(columns=["_merge"]).rename(columns={"brand_x": "brand", "brand_y": "traficom_brand"})
merged_df_final = merged_df_final.drop(columns=[col for col in ["traficom_brand"] if col in merged_df_final.columns])

model_feature_exclusions = {"model_merge_key", "model_family_clean", "model_looks_like_part_taxonomy"}
model_feature_columns = [
    col for col in merged_df_final.columns if col.startswith("model_") and col not in model_feature_exclusions
]
brand_feature_columns = [
    col for col in merged_df_final.columns if col.startswith("brand_") and col not in {"brand_merge_key", "brand_is_known_model_family"}
]
price_columns = [
    col for col in [
        "product_id", "part_name", "price", "quality_grade", "oem_number", "mileage",
        "brand", "model", "category", "subcategory", "scrape_date",
        "year_start", "year_end", "year_span", "year_mid",
    ] if col in merged_df_final.columns
]
merge_key_columns = [
    col for col in [
        "brand_merge_key", "model_merge_key", "repair_status",
        "brand_is_known_model_family", "model_looks_like_part_taxonomy", "model_family_clean",
    ] if col in merged_df_final.columns
]
other_columns = [
    col for col in merged_df_final.columns
    if col not in price_columns + merge_key_columns + model_feature_columns + brand_feature_columns
]
merged_df_final = merged_df_final[price_columns + merge_key_columns + other_columns + model_feature_columns + brand_feature_columns]

print(f"Final merged dataset shape after cleanup: {merged_df_final.shape}")
print("Final column cleanup completed.")
display(merged_df_final.head(10))


Final merged dataset shape after cleanup: (11374, 70)
Final column cleanup completed.


,product_id,part_name,price,quality_grade,oem_number,mileage,brand,model,category,subcategory,...,brand_ev_share,brand_hybrid_share,brand_turbo_share,brand_firstreg_total_2014_2026,brand_firstreg_recent_share,brand_firstreg_old_share,brand_firstreg_weighted_year,brand_firstreg_peak_year,brand_firstreg_peak_count,brand_firstreg_year_span
0,63136980.0,"Contact roll Airbag - , e-",177.6,A2,FI02042722A,224000.0,vw,golf,airbag,contact roll airbag,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
1,64390201.0,"Contact roll Airbag - , e-",177.6,A2,FI05351686A,272000.0,vw,golf,airbag,contact roll airbag,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
2,58952159.0,"Contact roll Airbag - , e-",177.6,A2,FI27837687A,134000.0,vw,golf,airbag,contact roll airbag,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
3,63225719.0,"Contact roll Airbag - , e-",177.6,A2,FI27837687A,253000.0,vw,golf,airbag,contact roll airbag,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
4,64336640.0,"Contact roll Airbag - , e-",177.6,A2,FI09389104A,152000.0,vw,golf,airbag,contact roll airbag,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
5,54655558.0,"Other airbags - , e-",23.7,A2,FI03986645A,<NA>,vw,golf,airbag,other airbags,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
6,54058388.0,"Other airbags - , e-",23.7,A2,FI10331575A,84000.0,vw,golf,airbag,other airbags,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
7,57846326.0,"Other airbags - , e-",17.8,A2,FI27837687A,227000.0,vw,golf,airbag,other airbags,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
8,53964588.0,"Other airbags - , e-",14.2,A2,FI06408963A,105000.0,vw,golf,airbag,other airbags,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0
9,54061937.0,"Passenger airbag - , e-",473.6,A1,FI10331575A,34000.0,vw,golf,airbag,passenger airbag,...,0.074727,0.073083,0.686131,100381.0,0.392644,0.390682,2019.842191,2018.0,13274.0,9.0


## Step 14 — Final validation checks


In [36]:
# Validate that the final merged dataset is complete and row-safe.
required_objects = ["price_df", "merged_df_final"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Step 14 depends on earlier merge cells. Rerun the notebook from Step 1 through Step 13 after the load and merge fixes. "
        f"Missing objects: {missing_objects}"
    )

final_row_count = len(merged_df_final)
final_column_count = merged_df_final.shape[1]
row_count_matches_price_df = final_row_count == len(price_df)
row_count_difference = final_row_count - len(price_df)

print(f"Final dataset shape: {merged_df_final.shape}")
print(f"Final row count: {final_row_count}")
print(f"Final column count: {final_column_count}")
print(f"Row count matches price_df: {row_count_matches_price_df}")
print(f"Row count difference vs price_df: {row_count_difference}")

if not row_count_matches_price_df:
    raise ValueError("The final merge changed the row count unexpectedly. Review duplicate keys before using this dataset.")

merge_validation_columns = [
    "brand_merge_key",
    "model_merge_key",
    "model_total_registered",
    "brand_total_registered",
    "model_firstreg_total_2014_2026",
    "brand_firstreg_total_2014_2026",
]

missing_merge_features_df = pd.DataFrame(
    {
        "column": merge_validation_columns,
        "missing_rows": [
            int(merged_df_final[column].isna().sum()) if column in merged_df_final.columns else pd.NA
            for column in merge_validation_columns
        ],
    }
)
display(missing_merge_features_df)

display(
    merged_df_final["repair_status"]
    .value_counts(dropna=False)
    .rename_axis("repair_status")
    .reset_index(name="row_count")
)

model_firstreg_match_count = int(merged_df_final["model_firstreg_total_2014_2026"].notna().sum()) if "model_firstreg_total_2014_2026" in merged_df_final.columns else pd.NA
brand_firstreg_match_count = int(merged_df_final["brand_firstreg_total_2014_2026"].notna().sum()) if "brand_firstreg_total_2014_2026" in merged_df_final.columns else pd.NA

coverage_summary_df = pd.DataFrame(
    {
        "feature_group": [
            "model-level market features",
            "brand-level market features",
            "model-level lifecycle features",
            "brand-level lifecycle features",
        ],
        "all_rows_covered": [
            bool(merged_df_final["model_total_registered"].notna().all()),
            bool(merged_df_final["brand_total_registered"].notna().all()),
            bool(merged_df_final["model_firstreg_total_2014_2026"].notna().all()) if "model_firstreg_total_2014_2026" in merged_df_final.columns else pd.NA,
            bool(merged_df_final["brand_firstreg_total_2014_2026"].notna().all()) if "brand_firstreg_total_2014_2026" in merged_df_final.columns else pd.NA,
        ],
        "missing_rows": [
            int(merged_df_final["model_total_registered"].isna().sum()),
            int(merged_df_final["brand_total_registered"].isna().sum()),
            int(merged_df_final["model_firstreg_total_2014_2026"].isna().sum()) if "model_firstreg_total_2014_2026" in merged_df_final.columns else pd.NA,
            int(merged_df_final["brand_firstreg_total_2014_2026"].isna().sum()) if "brand_firstreg_total_2014_2026" in merged_df_final.columns else pd.NA,
        ],
        "matched_rows": [
            int(merged_df_final["model_total_registered"].notna().sum()),
            int(merged_df_final["brand_total_registered"].notna().sum()),
            model_firstreg_match_count,
            brand_firstreg_match_count,
        ],
    }
)
display(coverage_summary_df)

model_firstreg_unmatched_pairs = (
    merged_df_final.loc[
        merged_df_final["model_firstreg_total_2014_2026"].isna(),
        ["brand_merge_key", "model_merge_key"],
    ]
    .dropna()
    .value_counts()
    .rename("row_count")
    .reset_index()
    .head(20)
    if "model_firstreg_total_2014_2026" in merged_df_final.columns else pd.DataFrame()
)
brand_firstreg_unmatched_keys = (
    merged_df_final.loc[
        merged_df_final["brand_firstreg_total_2014_2026"].isna(),
        ["brand_merge_key"],
    ]
    .dropna()
    .value_counts()
    .rename("row_count")
    .reset_index()
    .head(20)
    if "brand_firstreg_total_2014_2026" in merged_df_final.columns else pd.DataFrame()
)

print("Top unmatched model registration keys:")
print(model_firstreg_unmatched_pairs.to_string(index=False))
print("\nTop unmatched brand registration keys:")
print(brand_firstreg_unmatched_keys.to_string(index=False))

if previous_registration_coverage is not None:
    coverage_comparison_df = pd.DataFrame(
        {
            "metric": [
                "model first-registration matched rows",
                "brand first-registration matched rows",
                "final merged row count",
            ],
            "previous_saved_output": [
                previous_registration_coverage.get("previous_model_firstreg_matches"),
                previous_registration_coverage.get("previous_brand_firstreg_matches"),
                previous_registration_coverage.get("previous_row_count"),
            ],
            "current_output": [
                model_firstreg_match_count,
                brand_firstreg_match_count,
                final_row_count,
            ],
        }
    )
    coverage_comparison_df["delta"] = coverage_comparison_df["current_output"] - coverage_comparison_df["previous_saved_output"]
    display(coverage_comparison_df)

sample_columns = [
    col for col in [
        "product_id", "part_name", "brand", "model", "brand_merge_key",
        "model_merge_key", "repair_status", "model_total_registered", "brand_total_registered",
        "model_firstreg_total_2014_2026", "brand_firstreg_total_2014_2026",
    ] if col in merged_df_final.columns
]
display(merged_df_final[sample_columns].head(10))


Final dataset shape: (11374, 70)
Final row count: 11374
Final column count: 70
Row count matches price_df: True
Row count difference vs price_df: 0


,column,missing_rows
0,brand_merge_key,0
1,model_merge_key,0
2,model_total_registered,0
3,brand_total_registered,0
4,model_firstreg_total_2014_2026,0
5,brand_firstreg_total_2014_2026,0


,repair_status,row_count
0,original_valid,11374


,feature_group,all_rows_covered,missing_rows,matched_rows
0,model-level market features,True,0,11374
1,brand-level market features,True,0,11374
2,model-level lifecycle features,True,0,11374
3,brand-level lifecycle features,True,0,11374


Top unmatched model registration keys:
Empty DataFrame
Columns: [brand_merge_key, model_merge_key, row_count]
Index: []

Top unmatched brand registration keys:
Empty DataFrame
Columns: [brand_merge_key, row_count]
Index: []


,metric,previous_saved_output,current_output,delta
0,model first-registration matched rows,11374,11374,0
1,brand first-registration matched rows,11374,11374,0
2,final merged row count,11374,11374,0


,product_id,part_name,brand,model,brand_merge_key,model_merge_key,repair_status,model_total_registered,brand_total_registered,model_firstreg_total_2014_2026,brand_firstreg_total_2014_2026
0,63136980.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
1,64390201.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
2,58952159.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
3,63225719.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
4,64336640.0,"Contact roll Airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
5,54655558.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
6,54058388.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
7,57846326.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
8,53964588.0,"Other airbags - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0
9,54061937.0,"Passenger airbag - , e-",vw,golf,volkswagen,golf,original_valid,86762,279311,31752.0,100381.0


## Step 15 — Save final merged dataset


In [37]:
# Save the final merged dataset for the next analysis stage.
output_dir = base_dir / "datasets/merged"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "price_traficom_merged.csv"

merged_df_final.to_csv(output_path, index=False)
print(f"Saved merged dataset to: {output_path}")

reloaded_merged_df = pd.read_csv(output_path)
print(f"Reloaded merged dataset shape: {reloaded_merged_df.shape}")

Saved merged dataset to: /Users/riteshbhandari/Documents/Dokumentit – Ritesh - MacBook Pro/GitHub/DPPM/datasets/merged/price_traficom_merged.csv
Reloaded merged dataset shape: (11374, 70)


## Step 16 — Short final notebook conclusion
- Merge preparation was completed earlier in this notebook using cleaned Traficom summary tables and repaired price-data keys.
- The actual merge has now been performed using `brand_merge_key` and `model_merge_key`, not the raw source fields.
- Row count was preserved through both merge steps, and the merged dataset was saved successfully.
- The combined dataset is now ready for downstream analysis and modeling.
